In [2]:
!"C:/Users/alex/AppData/Local/Programs/Python/Python312/python.exe" -m pip install sacrebleu

zsh:1: no such file or directory: C:/Users/alex/AppData/Local/Programs/Python/Python312/python.exe


In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
import torch
import torch.nn as nn
import pandas as pd
import sentencepiece
from tqdm import tqdm
import comet_ml
import sacrebleu
from torch.utils.data import DataLoader

In [5]:
from dataset import WordTokenizer, TextDataset

In [6]:
train_dataset = TextDataset("data/train.de-en.de", "data/train.de-en.en", min_samples=30)
val_dataset = TextDataset("data/val.de-en.de", "data/val.de-en.en", tokenizers=(train_dataset.tokenizer_de, train_dataset.tokenizer_en))

In [7]:
# train_data, _ = train_test_split(dataset, test_size=0.7)

In [8]:
from model import LanguageModel

In [9]:
from train import train

In [ ]:
train_loader = DataLoader(train_dataset, 128, True)
val_loader = DataLoader(val_dataset, 128, True)

In [18]:
model = LanguageModel(train_dataset, embed_size=2, hidden_size=2, rnn_layers=1, rnn_type=nn.LSTM)
model.to("mps")           

LanguageModel(
  (embedding_en): Embedding(6136, 2, padding_idx=0)
  (embedding_de): Embedding(6494, 2, padding_idx=0)
  (rnn_encoder): LSTM(2, 2, batch_first=True)
  (rnn_decoder): LSTM(4, 2, batch_first=True)
  (linear): Linear(in_features=2, out_features=6136, bias=True)
)

In [19]:
optimizer   = torch.optim.Adam(model.parameters(), 1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, 15)
   
train(model, optimizer, train_loader=train_loader, scheduler=scheduler, val_loader=val_loader, num_epochs=15 )

Training 1/15:   0%|          | 27/6123 [00:11<43:15,  2.35it/s] 


KeyboardInterrupt: 

In [ ]:
torch.save(model.state_dict(), "check_lstm.pth")

In [ ]:
cnt = 0
with open("data/test1.de-en.de", "r", encoding="utf-8") as f_in:
    with open("data/test1_lstm.de-en.en", "w", encoding="utf-8") as f_out:
        for line in f_in.readlines():
            new_line = model.inference(line, temp=0.3)
            if new_line[-1] != "\n":
                new_line += "\n"
            f_out.write(new_line)
            cnt += 1